In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, GATConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import numpy as np
import pandas as pd
from tqdm import tqdm
import shutil
import concurrent.futures
import threading
from collections import defaultdict
import time
import copy

data_lock = threading.Lock()

def set_seed(seed: int):
    """Set random generators used by teacher training."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class GATTeacher(nn.Module):
    """GAT regressor used as a chemistry-stratified teacher."""
    def __init__(self, node_dim, edge_dim, global_dim, hidden_dims, dropout=0.2, gat_heads=4):
        super().__init__()
        if not hidden_dims:
            raise ValueError("hidden_dims must not be empty")
        if any(h_dim % gat_heads != 0 for h_dim in hidden_dims):
            raise ValueError("Each hidden dimension must be divisible by gat_heads")

        self.gat_heads = gat_heads
        self.node_norm = nn.BatchNorm1d(node_dim)
        self.edge_norm = nn.BatchNorm1d(edge_dim) if edge_dim else None
        self.global_norm = nn.BatchNorm1d(global_dim) if global_dim else None
        self.global_mlp = None
        if global_dim:
            self.global_mlp = nn.Sequential(
                nn.Linear(global_dim, hidden_dims[-1]),
                nn.ReLU(),
                nn.Dropout(dropout)
            )

        self.convs = nn.ModuleList()
        self.conv_norms = nn.ModuleList()
        in_dim = node_dim
        for h_dim in hidden_dims:
            self.convs.append(GATConv(
                in_channels=in_dim,
                out_channels=h_dim // gat_heads,
                heads=gat_heads,
                concat=True,
                dropout=dropout,
                edge_dim=edge_dim if edge_dim else None,
                add_self_loops=True
            ))
            self.conv_norms.append(nn.LayerNorm(h_dim))
            in_dim = h_dim

        self.dropout = nn.Dropout(dropout)
        self.final_dim = hidden_dims[-1] * (2 if global_dim else 1)
        self.output_mlp = nn.Sequential(
            nn.Linear(self.final_dim, self.final_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.final_dim // 2, 1)
        )

    def forward(self, data, return_feat=False):
        x = self.node_norm(data.x)
        edge_attr = getattr(data, 'edge_attr', None)
        if self.edge_norm is not None and edge_attr is not None:
            edge_attr = self.edge_norm(edge_attr)

        u = getattr(data, 'u', None)
        if self.global_norm is not None and u is not None:
            u = self.global_norm(u)

        for conv, conv_norm in zip(self.convs, self.conv_norms):
            x = conv(x, data.edge_index, edge_attr=edge_attr)
            x = conv_norm(x)
            x = F.elu(x)
            x = self.dropout(x)

        node_pool = global_mean_pool(x, data.batch)
        h = torch.cat([node_pool, self.global_mlp(u)], dim=1) if u is not None else node_pool
        out = self.output_mlp(h).squeeze(-1)
        return (out, h) if return_feat else out

def load_graphs(path: str):
    """Load graph dictionaries from a graph_data.pt file or directory."""
    print(f"Loading graph data: {path}")
    data = torch.load(os.path.join(path, 'graph_data.pt'))
    print(f"Samples: {len(data)} | node features: {data[0]['x'].shape[1]}")
    return data

def make_loader(graphs, batch_size, shuffle=True):
    """Build a PyG DataLoader from stored graph dictionaries."""
    return DataLoader([Data(**g) for g in graphs], batch_size=batch_size, shuffle=shuffle)

def train_and_eval(graphs, split_seed, learn_seed, dropout, batch_size, epochs=200, patience=50, gat_heads=4):
    """Train one teacher candidate and return its validation performance."""
    print(f"Teacher run | split_seed={split_seed}, learn_seed={learn_seed}, dropout={dropout}, batch_size={batch_size}")

    graphs = [{**g, 'y': g['y'].clone()} for g in graphs]

    labels = [g['label'].item() for g in graphs]
    
    try:
        
        idx_train, idx_val = train_test_split(
            np.arange(len(graphs)), 
            test_size=0.1, 
            random_state=split_seed, 
            shuffle=True,
            stratify=labels
        )
        print("Using stratified train/validation split.")
    except ValueError as e:
        
        print(f"Stratified split unavailable: {e}")
        print("Using an unstratified split instead.")
        idx_train, idx_val = train_test_split(
            np.arange(len(graphs)), 
            test_size=0.1, 
            random_state=split_seed, 
            shuffle=True
        )
    
    train_graphs = [graphs[i] for i in idx_train]
    val_graphs = [graphs[i] for i in idx_val]

    total_samples = len(graphs)
    train_samples = len(train_graphs)
    val_samples = len(val_graphs)

    from collections import Counter
    train_label_dist = Counter([g['label'].item() for g in train_graphs])
    val_label_dist = Counter([g['label'].item() for g in val_graphs])
    
    print("\nTeacher split summary:")
    print(f"Total samples: {total_samples}")
    print(f"Training samples: {train_samples} ({train_samples/total_samples:.1%})")
    print(f"Validation samples: {val_samples} ({val_samples/total_samples:.1%})")
    
    print("\nTraining stratum counts:")
    for label, count in sorted(train_label_dist.items()):
        print(f"  Stratum {label}: {count} samples ({count/train_samples:.1%})")
    
    print("\nValidation stratum counts:")
    for label, count in sorted(val_label_dist.items()):
        print(f"  Stratum {label}: {count} samples ({count/val_samples:.1%})")

    ys = torch.stack([g['y'] for g in train_graphs])
    y_mean, y_std = ys.mean().item(), ys.std().item() + 1e-8
    for g in train_graphs + val_graphs:
        g['y'] = (g['y'] - y_mean) / y_std

    train_loader = make_loader(train_graphs, batch_size)
    val_loader = make_loader(val_graphs, batch_size, shuffle=False)

    set_seed(learn_seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    sample = train_graphs[0]
    node_dim = sample['x'].size(1)
    edge_dim = sample['edge_attr'].size(1) if sample.get('edge_attr', None) is not None else 0
    global_dim = sample['u'].size(1) if sample.get('u', None) is not None else 0
    model = GATTeacher(node_dim, edge_dim, global_dim, hidden_dims=[128, 128], dropout=dropout, gat_heads=gat_heads).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.MSELoss()

    best_r2, best_state = -np.inf, None
    no_improve_epochs = 0

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses, train_preds, train_trues = [], [], []
        for batch in train_loader:
            batch = batch.to(device)
            pred = model(batch)
            loss = criterion(pred, batch.y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
            train_preds.append(pred.detach().cpu().numpy())
            train_trues.append(batch.y.cpu().numpy())
        train_r2 = r2_score(np.concatenate(train_trues), np.concatenate(train_preds))
        train_loss = np.mean(train_losses)

        model.eval()
        val_losses, val_preds, val_trues = [], [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                pred = model(batch)
                val_losses.append(criterion(pred, batch.y).item())
                val_preds.append(pred.cpu().numpy())
                val_trues.append(batch.y.cpu().numpy())
        val_r2 = r2_score(np.concatenate(val_trues), np.concatenate(val_preds))
        val_loss = np.mean(val_losses)

        if epoch % 10 == 0 or epoch == 1 or epoch == epochs:
            print(f"Epoch {epoch}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
                  f"Train R²: {train_r2:.4f} | Val R²: {val_r2:.4f}")

        if val_r2 > best_r2:
            best_r2 = val_r2
            best_state = copy.deepcopy(model.state_dict())
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            if no_improve_epochs >= patience:
                print(f"Early stopping: validation R2 did not improve for {patience} epochs.")
                break

    print(f"Best validation R2: {best_r2:.4f}\n")
    return best_r2, best_state, y_mean, y_std, node_dim, edge_dim, global_dim

def generate_predictions(model, graphs, y_mean, y_std, batch_size=64):
    """Generate normalized and original-scale predictions."""
    """Return predictions in teacher-normalized and original target units."""
    print("Generating teacher predictions...")
    loader = make_loader(graphs, batch_size, shuffle=False)
    device = next(model.parameters()).device
    model.eval()
    normalized_preds = []  
    denormalized_preds = []  
    
    for batch in tqdm(loader, desc="Predicting"):
        batch = batch.to(device)
        with torch.no_grad():
            
            p_normalized = model(batch).cpu().numpy()
            
            p_denormalized = p_normalized * y_std + y_mean
            
            normalized_preds.extend(p_normalized)
            denormalized_preds.extend(p_denormalized)
    
    return np.array(normalized_preds), np.array(denormalized_preds)

def process_feature_directory(feat_dir, predict_dir, output_dir, epochs, split_seeds, learn_seeds, dropouts, batch_sizes, 
                              all_normalized_labels, all_denormalized_labels):
    """Search teacher settings for one stratum, save the best model, and export soft labels."""
    """Train one stratified teacher and export its soft labels."""
    
    print(f"\nTraining stratum: {feat_dir}")
    start_time = time.time()
    
    try:
        graphs = load_graphs(feat_dir)
        feat_name = os.path.basename(feat_dir.rstrip('/\\'))
        feat_output_dir = os.path.join(output_dir, feat_name)
        os.makedirs(feat_output_dir, exist_ok=True)

        best_cfg, best_state, best_mean, best_std = None, None, None, None
        best_r2 = -np.inf
        best_node_dim = None
        best_edge_dim = None
        best_global_dim = None
        best_hidden_dims = [128, 128]  
        best_dropout = None

        for ss in split_seeds:
            for ls in learn_seeds:
                for do in dropouts:
                    for bs in batch_sizes:
                        print(f"[{feat_name}] Configuration: split_seed={ss}, learn_seed={ls}, dropout={do}, batch_size={bs}")
                        r2, state, y_m, y_s, node_dim, edge_dim, global_dim = train_and_eval(
                            graphs, ss, ls, do, bs, epochs
                        )
                        model_name = f"model_ss{ss}_ls{ls}_do{do}_bs{bs}.pt"
                        save_path = os.path.join(feat_output_dir, model_name)

                        torch.save({
                            'model_state_dict': state,
                            'node_dim': node_dim,
                            'edge_dim': edge_dim,
                            'global_dim': global_dim,
                            'hidden_dims': [128, 128],  
                            'dropout': do,
                            'gat_heads': 4,
                            'model_type': 'gat',        
                            'y_mean': y_m,              
                            'y_std': y_s                
                        }, save_path)
                        
                        if r2 > best_r2:
                            best_r2 = r2
                            best_cfg = (ss, ls, do, bs)
                            best_state = state
                            best_mean = y_m
                            best_std = y_s
                            best_node_dim = node_dim
                            best_edge_dim = edge_dim
                            best_global_dim = global_dim
                            best_dropout = do

        ss, ls, do, bs = best_cfg
        best_name = f"{feat_name}_model_ss{ss}_ls{ls}_do{do}_bs{bs}_best.pt"
        best_path = os.path.join(feat_output_dir, best_name)

        torch.save({
            'model_state_dict': best_state,
            'node_dim': best_node_dim,
            'edge_dim': best_edge_dim,
            'global_dim': best_global_dim,
            'hidden_dims': best_hidden_dims,
            'dropout': best_dropout,
            'gat_heads': 4,
            'model_type': 'gat',
            'y_mean': best_mean,
            'y_std': best_std
        }, best_path)

        best_models_dir = os.path.join(output_dir, "best_models")
        os.makedirs(best_models_dir, exist_ok=True)
        best_model_dest = os.path.join(best_models_dir, best_name)
        shutil.copyfile(best_path, best_model_dest)
        print(f"[{feat_name}] Best checkpoint copied to: {best_model_dest}")

        graphs_pred = load_graphs(predict_dir)

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = GATTeacher(
            node_dim=best_node_dim,
            edge_dim=best_edge_dim,
            global_dim=best_global_dim,
            hidden_dims=best_hidden_dims,
            dropout=best_dropout,
            gat_heads=4
        ).to(device)
        
        model.load_state_dict(best_state)
        
        normalized_preds, denormalized_preds = generate_predictions(
            model, graphs_pred, best_mean, best_std
        )

        normalized_df = pd.DataFrame({f"{feat_name}_teacher_normalized": normalized_preds})
        normalized_csv = os.path.join(feat_output_dir, 'predict_normalized_labels.csv')
        normalized_df.to_csv(normalized_csv, index=False)
        
        denormalized_df = pd.DataFrame({f"{feat_name}_teacher_denormalized": denormalized_preds})
        denormalized_csv = os.path.join(feat_output_dir, 'predict_denormalized_labels.csv')
        denormalized_df.to_csv(denormalized_csv, index=False)

        with data_lock:
            all_normalized_labels[feat_name] = normalized_preds
            all_denormalized_labels[feat_name] = denormalized_preds
            
        print(f"[{feat_name}] Completed in {time.time()-start_time:.2f} seconds.")
        print(f"  Normalized soft labels: {normalized_csv}")
        print(f"  Original-scale soft labels: {denormalized_csv}")
        
        return True, feat_name
    
    except Exception as e:
        print(f"[{feat_dir}] Failed: {e}")
        import traceback
        traceback.print_exc()
        return False, os.path.basename(feat_dir)

def main(base_dir: str, predict_dir: str, output_dir: str, epochs=200):
    """Train all stratified teachers and merge soft labels in a fixed teacher order."""
    os.makedirs(output_dir, exist_ok=True)
    print(f"Stratified feature root: {base_dir}")
    print(f"Prediction features: {predict_dir}")
    print(f"Output directory: {output_dir}")

    feat_dirs = [os.path.join(base_dir, d) for d in os.listdir(base_dir)
                 if os.path.isdir(os.path.join(base_dir, d))]
    num_features = len(feat_dirs)
    print(f"Detected {num_features} stratified directories.\n")

    all_normalized_labels = defaultdict(list)
    all_denormalized_labels = defaultdict(list)

    seeds = [0, 8, 42, 100, 456, 618, 1189, 2025, 2077, 2048]
    split_seeds = seeds
    learn_seeds = seeds
    dropouts = [0.1, 0.2, 0.3]
    batch_sizes = [32, 64]

    successful_features = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=min(num_features, os.cpu_count())) as executor:
        
        future_to_feat = {
            executor.submit(
                process_feature_directory, 
                feat_dir, 
                predict_dir, 
                output_dir, 
                epochs,
                split_seeds,
                learn_seeds,
                dropouts,
                batch_sizes,
                all_normalized_labels,
                all_denormalized_labels
            ): os.path.basename(feat_dir)
            for feat_dir in feat_dirs
        }

        with tqdm(total=len(future_to_feat), desc="Training teachers") as pbar:
            for future in concurrent.futures.as_completed(future_to_feat):
                feat_name = future_to_feat[future]
                try:
                    success, name = future.result()
                    if success:
                        successful_features.append(name)
                        pbar.update(1)
                        pbar.set_postfix_str(f"Completed: {name}")
                except Exception as e:
                    print(f"Error for {feat_name}: {e}")
                    pbar.update(1)

    teacher_priority = ('qcut', 'elem', 'molwt', 'fp', 'scaffold')

    def teacher_sort_key(name):
        for order, token in enumerate(teacher_priority):
            if token in name:
                return order, name
        return len(teacher_priority), name

    ordered_names = sorted(all_normalized_labels.keys(), key=teacher_sort_key)
    normalized_df = pd.DataFrame({name: all_normalized_labels[name] for name in ordered_names})
    denormalized_df = pd.DataFrame({name: all_denormalized_labels[name] for name in ordered_names})
    print(f"Teacher/soft-label order: {ordered_names}")

    all_normalized_path = os.path.join(output_dir, "all_normalized_labels.csv")
    normalized_df.to_csv(all_normalized_path, index=False)
    
    all_denormalized_path = os.path.join(output_dir, "all_denormalized_labels.csv")
    denormalized_df.to_csv(all_denormalized_path, index=False)
    
    print(f"\nCompleted {len(successful_features)}/{num_features} strata.")
    print(f"Normalized soft labels: {all_normalized_path}")
    print(f"Original-scale soft labels: {all_denormalized_path}")

    return all_normalized_path, all_denormalized_path

if __name__ == "__main__":
    # Set these directories before running.
    base_dir = None
    predict_dir = None
    output_dir = None
    if not all([base_dir, predict_dir, output_dir]):
        raise ValueError("Set base_dir, predict_dir, and output_dir before running.")

    main(base_dir, predict_dir, output_dir, epochs=300)
